In [ ]:
%matplotlib widget
import os
import sys
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torchkbnufft as tkbn
import pypulseq as pp
import MRzeroCore as mr0
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.checkpoint import checkpoint

def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for d in [cur, *cur.parents]:
        if (d / "Modules").is_dir():
            return d
    raise RuntimeError(f"Could not find a folder containing 'Modules' from {cur}")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Drop any cached Modules* modules so a kernel restart is only needed for
# *new* modules, not for edits to existing ones - re-running this cell
# always re-imports the current Modules/ code.
for _m in [m for m in list(sys.modules) if m == 'Modules' or m.startswith('Modules.')]:
    del sys.modules[_m]

from Modules import PARAMS, build_sim_data
from Modules.bssfp_params import TISSUES, T2STAR, NAMES
from Modules.recon_ops import LinearOperator, ChannelOperator, IdentityOperator, recon_nufft, recon_cg, signed_series, mag_series, get_partition, K3, A_ops, _sub
from Modules.Plan import KbNufft, KbNufftAdjoint
from Modules.NumSolvers import conjugate_gradient
from Modules.kwic_frames import build_kwic
from Modules.Regularizer import Regularizer 
from Modules.DeepADMM import _admm_iter
from Modules import setup_recon

torch.set_default_device("cuda")
device = torch.device("cuda")

In [ ]:
model_name = "zs3d_frames.pt"
model_file = REPO_ROOT / "Training" / "Trained models" / model_name

In [ ]:
zs3d = torch.load(model_file)

In [ ]:
mid = n // 2
fs = np.unique(np.linspace(0, F_ALL - 75, 12).astype(int))
vmax = float(np.percentile(np.concatenate([
zs3d[int(f)].abs().cpu().numpy().ravel() for f in fs]), 99.9))
#vmax = max(vmax, float(np.percentile(np.concatenate([
#regrid[int(f)].abs().cpu().numpy().ravel() for f in fs]), 99.9)))
fig, axes = plt.subplots(len(fs), 2, figsize=(3.6 * 2, 3.0 * len(fs)))
for i, f in enumerate(fs):
    sl_zs = np.rot90(zs3d[int(f)].abs().cpu().numpy()[:,:, mid].T, 2)
    sl_rg = np.rot90(regrid[int(f)].abs().cpu().numpy()[:,:, mid].T, 2)
    axes[i, 0].imshow(sl_zs, cmap="gray", vmin=0)
    axes[i, 1].imshow(sl_rg, cmap="gray", vmin=0)
    axes[i, 0].set_ylabel(f"TI={ti_frames[int(f)]*1e3:4.0f} ms", fontsize=8)
for a in axes[i]:
    a.axis("off")
axes[0, 0].set_title("ZS-SSL (inference)", fontsize=9)
axes[0, 1].set_title("regrid (no net)", fontsize=9)
plt.tight_layout(); plt.show()